<a href="https://colab.research.google.com/github/nhentze1/Minimum-Wages-and-Business-Models-Company-Owned-vs.-Franchise-Restaurants/blob/main/minwageDiD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Minimum Wages and Restaurant Business Models

This notebook replicates an employment benchmark from Card and Krueger (1994) and extends the analysis to ownership-specific employment, prices, and daily opening hours.

## Data Access and Setup

Use a Python 3.12 environment as the notebook kernel. The first code cell installs the required packages into that kernel. It loads **CardKrueger_1994_Fastfood.xlsx** from the working directory when available; otherwise it downloads the workbook from the public GitHub repository.

Run all cells in order. Package installation requires internet access; the analysis uses the local workbook when it is present.

## Load Libraries and Data

The setup cell uses `%pip` so installation targets the selected notebook kernel. Data-loading errors stop execution rather than leaving undefined analysis objects.

In [18]:
%pip install pandas numpy matplotlib seaborn openpyxl statsmodels -q
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
from IPython.display import display, Markdown

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 20)
pd.set_option("display.max_colwidth", None)

url_to_excel = "https://raw.githubusercontent.com/nhentze1/Minimum-Wages-and-Business-Models-Company-Owned-vs.-Franchise-Restaurants/main/CardKrueger_1994_Fastfood.xlsx"
local_file = Path("CardKrueger_1994_Fastfood.xlsx")
data_source = local_file if local_file.exists() else url_to_excel
excel_file = pd.ExcelFile(data_source)
df = pd.read_excel(excel_file, sheet_name="Daten")
print(f"Loaded {len(df)} restaurants from {data_source}.")
display(df.head())

Loaded 410 restaurants from https://raw.githubusercontent.com/nhentze1/Minimum-Wages-and-Business-Models-Company-Owned-vs.-Franchise-Restaurants/main/CardKrueger_1994_Fastfood.xlsx.


,sheet,chain,co_owned,state,southj,centralj,northj,pa1,pa2,shore,...,firstin2,special2,meals2,open2r,hrsopen2,psoda2,pfry2,pentree2,nregs2,nregs112
0,46,1,0,0,0,0,0,1,0,0,...,0.08,1.0,2.0,6.5,16.5,1.03,NaN,0.94,4.0,4.0
1,49,2,0,0,0,0,0,1,0,0,...,0.05,0.0,2.0,10.0,13.0,1.01,0.89,2.35,4.0,4.0
2,506,2,1,0,0,0,0,1,0,0,...,0.25,NaN,1.0,11.0,11.0,0.95,0.74,2.33,4.0,3.0
3,56,4,1,0,0,0,0,1,0,0,...,0.15,0.0,2.0,10.0,12.0,0.92,0.79,0.87,2.0,2.0
4,61,4,1,0,0,0,0,1,0,0,...,0.15,0.0,2.0,10.0,12.0,1.01,0.84,0.95,2.0,2.0


## Workbook Structure and English Variable Guide

The workbook contains data, source information, and a codebook. The guide below documents the analysis variables in English without reproducing the workbook's German metadata. Original variable and sheet identifiers are retained for reproducibility.

In [19]:
# Show the names of all sheets in the file
if 'excel_file' in globals():
    print("Available sheets in the file:", excel_file.sheet_names)
else:
    print("Please run the setup cells above first.")

Available sheets in the file: ['Info', 'Daten', 'Codebuch']


In [20]:
variable_definitions = pd.DataFrame([
    ("sheet", "Unique restaurant ID"),
    ("state", "1 = New Jersey; 0 = Pennsylvania"),
    ("co_owned", "1 = Company-owned; 0 = Franchise"),
    ("chain", "1 = Burger King; 2 = KFC; 3 = Roy Rogers; 4 = Wendy's"),
    ("empft / emppt / nmgrs", "Full-time workers / part-time workers / managers"),
    ("psoda / pfry / pentree", "Tax-inclusive price: medium soda / small fries / main entree (USD)"),
    ("hrsopen", "Daily opening hours"),
    ("wage_st", "Starting hourly wage (USD)"),
    ("status2", "0 = Refused; 1 = Interviewed; 2 = Renovation; 3 = Permanently closed; 4 = Road construction; 5 = Fire"),
    ("Suffix 2", "Follow-up survey; no suffix denotes baseline"),
], columns=["Variable", "English definition"])
display(variable_definitions)

,Variable,English definition
0,sheet,Unique restaurant ID
1,state,1 = New Jersey; 0 = Pennsylvania
2,co_owned,1 = Company-owned; 0 = Franchise
3,chain,1 = Burger King; 2 = KFC; 3 = Roy Rogers; 4 = Wendy's
4,empft / emppt / nmgrs,Full-time workers / part-time workers / managers
5,psoda / pfry / pentree,Tax-inclusive price: medium soda / small fries / main entree (USD)
6,hrsopen,Daily opening hours
7,wage_st,Starting hourly wage (USD)
8,status2,0 = Refused; 1 = Interviewed; 2 = Renovation; 3 = Permanently closed; 4 = Road construction; 5 = Fire
9,Suffix 2,Follow-up survey; no suffix denotes baseline


### Setup Completed Successfully

The data is loaded and stored in the variable `df`. We can now proceed with the exploratory data analysis and Difference-in-Differences (DiD) estimations.

In [21]:
# Confirming that the data is ready
if 'df' in globals():
    print(f"Ready for analysis! The dataset contains {df.shape[0]} rows and {df.shape[1]} columns.")
else:
    print("Please run the setup cells above to load the data.")

Ready for analysis! The dataset contains 410 rows and 46 columns.


## Research Design and Outcome Definitions

On April 1, 1992, New Jersey increased its minimum wage from USD 4.25 to USD 5.05 per hour; Pennsylvania retained USD 4.25. The first survey was in February/March and the second in November/December 1992.

We compare before-to-after changes in New Jersey with changes in Pennsylvania, separately for franchise and company-owned restaurants. The ownership contrast is a difference-in-difference-in-differences (DDD): franchise DiD minus company-owned DiD.

- **Employment:** full-time workers + managers + 0.5 x part-time workers, in full-time equivalents (FTE). A missing component makes the total missing. Permanently closed restaurants have zero follow-up employment; temporary closures and refusals are not automatically set to zero.
- **Prices:** the sum of a medium soda, small fries, and main entree, in nominal tax-inclusive USD. All three components must be observed. This is a fixed item basket, not a transaction-weighted price index.
- **Operations:** reported daily opening hours. Main price and hours estimates require an interviewed restaurant at follow-up; they describe the observed operating sample rather than all original restaurants.

Each outcome uses its own sample with both waves observed. We report sample counts and attrition rather than treating missing values as zeros. Ownership and chain are measured at baseline; ownership transitions are not observed.

In [22]:
analysis = df.copy()
required_columns = [
    "sheet", "chain", "state", "co_owned", "status2",
    "empft", "nmgrs", "emppt", "empft2", "nmgrs2", "emppt2",
    "psoda", "pfry", "pentree", "psoda2", "pfry2", "pentree2",
    "hrsopen", "hrsopen2", "wage_st", "wage_st2",
]
assert set(required_columns).issubset(analysis.columns), "Required variables are missing."
analysis[required_columns] = analysis[required_columns].apply(pd.to_numeric, errors="raise")
assert analysis["sheet"].is_unique, "Restaurant IDs must be unique."
assert analysis[["sheet", "chain", "state", "co_owned", "status2"]].notna().all().all()
assert analysis["state"].isin([0, 1]).all()
assert analysis["co_owned"].isin([0, 1]).all()
assert analysis["chain"].isin([1, 2, 3, 4]).all()
assert analysis["status2"].isin([0, 1, 2, 3, 4, 5]).all()

analysis["franchise"] = 1 - analysis["co_owned"]
analysis["state_name"] = analysis["state"].map({0: "Pennsylvania", 1: "New Jersey"})
analysis["ownership"] = analysis["franchise"].map({0: "Company-owned", 1: "Franchise"})
analysis["chain_name"] = analysis["chain"].map({1: "Burger King", 2: "KFC", 3: "Roy Rogers", 4: "Wendy's"})
status_names = {
    0: "Refused interview", 1: "Interviewed", 2: "Closed for renovation",
    3: "Permanently closed", 4: "Closed for road construction",
    5: "Closed after shopping-center fire",
}
analysis["follow_up_status"] = analysis["status2"].map(status_names)

for wave, suffix in [("before", ""), ("after", "2")]:
    employment_columns = [f"empft{suffix}", f"nmgrs{suffix}", f"emppt{suffix}"]
    price_columns = [f"psoda{suffix}", f"pfry{suffix}", f"pentree{suffix}"]
    analysis[f"employment_{wave}"] = analysis[employment_columns].mul([1, 1, 0.5]).sum(axis=1, min_count=3)
    analysis[f"meal_price_{wave}"] = analysis[price_columns].sum(axis=1, min_count=3)
    analysis[f"hours_{wave}"] = analysis[f"hrsopen{suffix}"]

analysis.loc[analysis["status2"].eq(3), "employment_after"] = 0.0
analysis.loc[analysis["status2"].ne(1), ["meal_price_after", "hours_after"]] = np.nan
outcomes = {
    "employment": "Employment (FTE)",
    "meal_price": "Meal basket price (USD)",
    "hours": "Daily opening hours",
}
samples = {}
for outcome in outcomes:
    before, after = f"{outcome}_before", f"{outcome}_after"
    values = analysis[[before, after]]
    assert (values.ge(0) | values.isna()).all().all(), f"Negative values in {outcome}."
    sample = analysis.dropna(subset=[before, after]).copy()
    sample["change"] = sample[after] - sample[before]
    assert sample.groupby(["state", "franchise"]).size().size == 4
    samples[outcome] = sample

benchmark = analysis.groupby("state_name")[["employment_before", "employment_after"]].mean()
expected = pd.DataFrame(
    {"employment_before": [20.44, 23.33], "employment_after": [21.03, 21.17]},
    index=["New Jersey", "Pennsylvania"],
)
assert np.allclose(benchmark.loc[expected.index], expected, atol=0.005), "Published employment benchmark does not match."
print("Employment benchmark reproduced using all available observations in each wave:")
display(benchmark.round(2))
print("Restaurants by state and baseline ownership:")
display(pd.crosstab(analysis["state_name"], analysis["ownership"]))
print("Follow-up status by state:")
display(pd.crosstab(analysis["follow_up_status"], analysis["state_name"]))
print("Paired sample sizes:")
display(pd.DataFrame({outcomes[outcome]: sample.groupby(["state_name", "ownership"]).size() for outcome, sample in samples.items()}))
print("Restaurants by chain, state, and ownership:")
display(pd.crosstab([analysis["chain_name"], analysis["state_name"]], analysis["ownership"]))

Employment benchmark reproduced using all available observations in each wave:


,employment_before,employment_after
state_name,,
New Jersey,20.44,21.03
Pennsylvania,23.33,21.17


Restaurants by state and baseline ownership:


ownership,Company-owned,Franchise
state_name,,
New Jersey,113,218
Pennsylvania,28,51


Follow-up status by state:


state_name,New Jersey,Pennsylvania
follow_up_status,,
Closed after shopping-center fire,1,0
Closed for renovation,2,0
Closed for road construction,1,0
Interviewed,321,78
Permanently closed,5,1
Refused interview,1,0


Paired sample sizes:


Employment (FTE)  Meal basket price (USD)  \
state_name   ownership                                                  
New Jersey   Company-owned               108                       92   
             Franchise                   201                      195   
Pennsylvania Company-owned                26                       26   
             Franchise                    49                       42   

                            Daily opening hours  
state_name   ownership                           
New Jersey   Company-owned                  108  
             Franchise                      212  
Pennsylvania Company-owned                   28  
             Franchise                       50

Restaurants by chain, state, and ownership:


ownership                 Company-owned  Franchise
chain_name  state_name                            
Burger King New Jersey               25        111
            Pennsylvania              0         35
KFC         New Jersey               28         40
            Pennsylvania              8          4
Roy Rogers  New Jersey               56         26
            Pennsylvania             12          5
Wendy's     New Jersey                4         41
            Pennsylvania              8          7

***Summary Statistics***

In [25]:
# 1. Distribution of Store Types
rows_1 = []

for chain, name in {
    1: "Burger King",
    2: "KFC",
    3: "Roy Rogers",
    4: "Wendy's",
}.items():
    rows_1.append(table2_row(name, f"chain_{chain}", percent=True))

rows_1.append(table2_row("Company-owned", "co_owned", percent=True))
table_1 = pd.DataFrame(rows_1)


# 2. Means in Wave 1
wave1 = [
    ("FTE employment", "fte1", False),
    ("Percentage full-time employees", "pct_full1", False),
    ("Starting wage", "wage_st", False),
    ("Wage = $4.25 (percentage)", "wage_425_1", True),
    ("Price of full meal", "meal_price1", False),
    ("Hours open (weekday)", "hrsopen", False),
    ("Recruiting bonus", "bonus", True),
]

rows_2 = [
    table2_row(label, variable, percent=percent)
    for label, variable, percent in wave1
]
table_2 = pd.DataFrame(rows_2)


# 3. Means in Wave 2
wave2 = [
    ("FTE employment", "fte2", False),
    ("Percentage full-time employees", "pct_full2", False),
    ("Starting wage", "wage_st2", False),
    ("Wage = $4.25 (percentage)", "wage_425_2", True),
    ("Wage = $5.05 (percentage)", "wage_505_2", True),
    ("Price of full meal", "meal_price2", False),
    ("Hours open (weekday)", "hrsopen2", False),
    ("Recruiting bonus", "special2", True),
]

rows_3 = [
    table2_row(label, variable, percent=percent)
    for label, variable, percent in wave2
]
table_3 = pd.DataFrame(rows_3)


# Display each section separately, in the paper's order.
number_format = {
    "NJ": "{:.1f}",
    "SE (NJ)": "({:.2f})",
    "PA": "{:.1f}",
    "SE (PA)": "({:.2f})",
    "t": "{:.1f}",
}

display(Markdown("### 1. Distribution of Store Types (percentages)"))
display(table_1.style.format(number_format).hide(axis="index"))

display(Markdown("### 2. Means in Wave 1"))
display(table_2.style.format(number_format).hide(axis="index"))

display(Markdown("### 3. Means in Wave 2"))
display(table_3.style.format(number_format).hide(axis="index"))

NameError: name 'table2_row' is not defined

## Descriptive Changes and Difference-in-Differences

The plots use the same paired restaurants before and after within each outcome, avoiding changes driven solely by a different observed sample in each wave. A two-point line is descriptive, not evidence that pre-treatment trends were parallel.

For each outcome, we estimate a first-difference regression:

$$\Delta Y_i = \alpha + \beta\,NJ_i + \gamma\,Franchise_i + \delta\,(NJ_i \times Franchise_i) + \varepsilon_i.$$

The company-owned DiD is $\beta$, the franchise DiD is $\beta + \delta$, and the ownership difference (DDD) is $\delta$. First-differencing removes time-invariant restaurant effects; this is the two-period counterpart of a restaurant-fixed-effects model with the relevant post-period interactions.

A negative DDD means a more negative franchise response in FTE or opening hours; a positive price DDD means a larger franchise price increase. These are relative effects: a negative DDD does not necessarily mean franchise employment or hours fell in absolute terms.

We report two-sided 95% HC3 heteroskedasticity-robust confidence intervals and unadjusted p-values. With one change observation per restaurant, HC3 treats restaurants as independent. **These intervals do not account for state-level common shocks and are not reliable policy-level inference with only two states.** Results are exploratory, including across the three outcomes.

In [ ]:
descriptive_tables = []
figure, axes = plt.subplots(1, 3, figsize=(16, 5.5), constrained_layout=True)
state_colors = {"New Jersey": "#166e67", "Pennsylvania": "#b44b47"}
ownership_styles = {"Company-owned": "--", "Franchise": "-"}
ownership_markers = {"Company-owned": "s", "Franchise": "o"}
for axis, (outcome, label) in zip(axes, outcomes.items()):
    sample = samples[outcome]
    grouped = sample.groupby(["state_name", "ownership"])
    summary = grouped.agg(
        restaurants=("sheet", "size"),
        before=(f"{outcome}_before", "mean"),
        after=(f"{outcome}_after", "mean"),
        change=("change", "mean"),
    )
    descriptive_tables.append(summary.assign(outcome=label).reset_index())
    for (state_name, ownership), values in summary.iterrows():
        axis.plot(
            [0, 1], [values["before"], values["after"]],
            marker=ownership_markers[ownership], color=state_colors[state_name],
            markerfacecolor="white" if ownership == "Company-owned" else state_colors[state_name],
            markersize=7, markeredgewidth=1.5, linewidth=2,
            linestyle=ownership_styles[ownership],
            label=f"{state_name}: {ownership}",
        )
    axis.set_xticks([0, 1], ["Before", "After"])
    axis.set_xlim(-0.15, 1.15)
    axis.set_title(label)
    axis.set_ylabel(label)
legend_handles, legend_labels = axes[0].get_legend_handles_labels()
figure.legend(
    legend_handles, legend_labels, loc="outside lower center", ncol=2,
    fontsize=10, handlelength=4, numpoints=2, columnspacing=2,
    frameon=False,
 )
display(pd.concat(descriptive_tables, ignore_index=True).set_index(["outcome", "state_name", "ownership"]).round(3))
plt.show()

def contrast_row(model, expression, outcome_label, effect_label, specification="Unadjusted"):
    contrast = model.t_test(expression)
    interval = np.asarray(contrast.conf_int()).reshape(-1)
    return {
        "Outcome": outcome_label,
        "Specification": specification,
        "Effect": effect_label,
        "Estimate": np.asarray(contrast.effect).item(),
        "Robust SE": np.asarray(contrast.sd).item(),
        "CI lower": interval[0],
        "CI upper": interval[1],
        "p-value": np.asarray(contrast.pvalue).item(),
        "Restaurants": int(model.nobs),
    }

models = {}
effect_rows = []
pooled_rows = []
manual_rows = []
for outcome, label in outcomes.items():
    sample = samples[outcome]
    model = smf.ols("change ~ state * franchise", data=sample).fit(cov_type="HC3")
    assert np.linalg.matrix_rank(model.model.exog) == model.model.exog.shape[1]
    models[outcome] = model
    for expression, effect_label in [
        ("state", "Company-owned DiD"),
        ("state + state:franchise", "Franchise DiD"),
        ("state:franchise", "Franchise minus company-owned (DDD)"),
    ]:
        effect_rows.append(contrast_row(model, expression, label, effect_label))
    means = sample.groupby(["state", "franchise"])["change"].mean()
    company_did = means.loc[(1, 0)] - means.loc[(0, 0)]
    franchise_did = means.loc[(1, 1)] - means.loc[(0, 1)]
    ddd = franchise_did - company_did
    assert np.isclose(model.params["state"], company_did)
    assert np.isclose(model.params["state"] + model.params["state:franchise"], franchise_did)
    assert np.isclose(model.params["state:franchise"], ddd)
    manual_rows.append({"Outcome": label, "Company-owned DiD": company_did, "Franchise DiD": franchise_did, "DDD": ddd})
    pooled = smf.ols("change ~ state", data=sample).fit(cov_type="HC3")
    pooled_rows.append(contrast_row(pooled, "state", label, "Pooled DiD"))

pooled_results = pd.DataFrame(pooled_rows)
results = pd.DataFrame(effect_rows)
print("Pooled DiD on outcome-specific paired samples (not an exact replication of every paper specification):")
display(pooled_results.drop(columns="Specification").round(4))
print("Manual group-mean calculations, verified against regression coefficients:")
display(pd.DataFrame(manual_rows).round(4))
print("Ownership-specific effects with exploratory restaurant-level HC3 uncertainty:")
display(results.drop(columns="Specification").round(4))
assert np.isfinite(results[["Estimate", "Robust SE", "CI lower", "CI upper", "p-value"]].to_numpy()).all()
print("All three manual DiD/DDD identities match the regression estimates.")

## Sensitivity Checks

1. **Chain-adjusted changes:** add chain indicators to the first-difference regression. These allow common chain-specific time shocks, not chain-specific treatment effects. They do not resolve missing ownership-by-state cells within a chain.
2. **Employment without permanent closures:** exclude permanently closed restaurants to show how the extensive margin affects the estimate. This changes the estimand and is not a correction for non-random survival.
3. **Opening hours including permanent closures:** assign zero follow-up hours to permanent closures only. Temporary closures, refusals, and other missing hours remain missing.
4. **Log meal-price changes:** estimate 100 times the change in log basket prices for strictly positive paired prices. Units are log points, approximately percentage points for small changes, rather than dollars.

The table reports only the franchise-minus-company DDD for each specification. Different samples, units, and assumptions mean these rows should not be pooled into one estimate.

In [ ]:
sensitivity_rows = results.loc[results["Effect"].eq("Franchise minus company-owned (DDD)")].to_dict("records")
for outcome, label in outcomes.items():
    adjusted = smf.ols("change ~ state * franchise + C(chain)", data=samples[outcome]).fit(cov_type="HC3")
    assert np.linalg.matrix_rank(adjusted.model.exog) == adjusted.model.exog.shape[1]
    sensitivity_rows.append(contrast_row(adjusted, "state:franchise", label, "Franchise minus company-owned (DDD)", "Chain-adjusted changes"))

employment_survivors = samples["employment"].loc[samples["employment"]["status2"].ne(3)].copy()
hours_with_closures = analysis.copy()
hours_with_closures.loc[hours_with_closures["status2"].eq(3), "hours_after"] = 0.0
hours_with_closures = hours_with_closures.dropna(subset=["hours_before", "hours_after"]).copy()
hours_with_closures["change"] = hours_with_closures["hours_after"] - hours_with_closures["hours_before"]
log_prices = samples["meal_price"].loc[
    samples["meal_price"][["meal_price_before", "meal_price_after"]].gt(0).all(axis=1)
].copy()
log_prices["change"] = 100 * (np.log(log_prices["meal_price_after"]) - np.log(log_prices["meal_price_before"]))

for sample, label, specification in [
    (employment_survivors, outcomes["employment"], "Exclude permanent closures"),
    (hours_with_closures, outcomes["hours"], "Permanent closures have zero hours"),
    (log_prices, "Meal basket price (100 log points)", "Positive paired prices"),
]:
    assert sample.groupby(["state", "franchise"]).size().size == 4
    sensitivity_model = smf.ols("change ~ state * franchise", data=sample).fit(cov_type="HC3")
    sensitivity_rows.append(contrast_row(sensitivity_model, "state:franchise", label, "Franchise minus company-owned (DDD)", specification))

sensitivity_results = pd.DataFrame(sensitivity_rows)
assert np.isfinite(sensitivity_results[["Estimate", "Robust SE", "CI lower", "CI upper", "p-value"]].to_numpy()).all()
display(sensitivity_results.drop(columns="Effect").round(4))

retention_rows = []
for outcome, label in outcomes.items():
    retained = analysis["sheet"].isin(samples[outcome]["sheet"])
    retention = analysis.assign(retained=retained).groupby(["state_name", "ownership"]).agg(
        original_restaurants=("sheet", "size"),
        paired_restaurants=("retained", "sum"),
    )
    retention["excluded_restaurants"] = retention["original_restaurants"] - retention["paired_restaurants"]
    retention["retention_percent"] = 100 * retention["paired_restaurants"] / retention["original_restaurants"]
    retention_rows.append(retention.assign(outcome=label).reset_index())
print("Outcome-specific retention by state and ownership:")
display(pd.concat(retention_rows, ignore_index=True).set_index(["outcome", "state_name", "ownership"]).round(1))

display(Markdown("## Results Summary"))
ddd_results = results.loc[results["Effect"].eq("Franchise minus company-owned (DDD)")]
summary_lines = []
for _, result in ddd_results.iterrows():
    interval_description = "includes zero" if result["CI lower"] <= 0 <= result["CI upper"] else "excludes zero"
    summary_lines.append(
        f"- **{result['Outcome']}:** franchise minus company-owned DiD = {result['Estimate']:+.3f}; "
        f"exploratory HC3 95% CI [{result['CI lower']:.3f}, {result['CI upper']:.3f}], which {interval_description}; "
        f"{int(result['Restaurants'])} paired restaurants."
    )
display(Markdown("\n".join(summary_lines)))
display(Markdown(
    "These estimates do not establish that the business models responded differently. "
    "An interval containing zero is not evidence that the effects are identical. "
    "Compare the sensitivity table before drawing substantive conclusions; "
    "restaurant-level uncertainty does not solve the two-state policy-inference problem."
))

## Interpretation and Limitations

- **Employment:** the unadjusted ownership contrast does not support a clear claim that franchise restaurants reduced employment more than company-owned restaurants. The wide interval also does not establish equivalence.
- **Prices:** the direction is opposite to a larger franchise price increase. The chain-adjusted DDD has an exploratory HC3 interval below zero, whereas the unadjusted and log-price contrasts include zero. The price finding is specification-sensitive, not a robust causal conclusion.
- **Opening hours:** the point estimate changes when permanent closures are assigned zero hours; the ownership contrasts remain imprecise. Neither specification establishes a clear differential response.
- **Identification:** a causal DiD interpretation requires comparable untreated trends within ownership groups, no anticipation, and no relevant spillovers. The ownership DDD further requires no untreated New Jersey-versus-Pennsylvania trend difference specific to ownership. Only one pre-treatment wave is available, so these assumptions cannot be tested with pre-trend or event-study evidence.
- **Inference:** treatment varies at the state level, with only one treated and one control state. Restaurant-level HC3 intervals do not capture common state shocks. Conventional two-cluster standard errors are not a credible fix. P-values are exploratory and not adjusted for multiple outcomes or specifications.
- **Composition and selection:** ownership is not randomized, chain composition differs, and some chain-by-state ownership cells are empty. Chain indicators allow common chain-specific changes but cannot eliminate all ownership-related confounding. Prices and hours condition on observed follow-up interviews; outcome-specific missingness may be selective.
- **Measurement and scope:** survey measurement error, nominal tax-inclusive prices, unobserved product quality or sales weights, and the short two-wave horizon limit interpretation. These results concern this historical fast-food sample, not all restaurants or minimum-wage settings.

## Reference

Card, D., and Krueger, A. B. (1994). *Minimum Wages and Employment: A Case Study of the Fast-Food Industry in New Jersey and Pennsylvania.* American Economic Review, 84(4), 772-793.

The repository workbook is a conversion of the authors' public data. The employment means are checked against the workbook's published-paper benchmark; the ownership, price-basket, and hours specifications are extensions rather than a complete replication of every table in the paper.

-----------------------------------------------------------------------------

```markdown
## Replication of Table 3 from Card and Krueger (1994)

We replicate Table 3 ("AVERAGE EMPLOYMENT PER STORE BEFORE AND AFTER THE RISE IN NEW JERSEY MINIMUM WAGE") using the criteria described in the paper:
- **FTE definition:** full-time + managers + 0.5 * part-time.
- **Subsample divisions for NJ** based on baseline starting wage (`wage_st`):
  - *Low*: $4.25 (exactly)
  - *Midrange*: $4.26 to $4.99
  - *High*: $\ge$ $5.00
- **Row 4 (Balanced sample):** Only restaurants with valid FTE in both wave 1 and wave 2.
- **Row 5 (Setting temporary closures to 0):** Temporarily closed restaurants (indicated by follow-up status codes such as renovation or other temporary issues) set to 0 rather than NaN in wave 2, using the balanced baseline sample.
```

In [ ]:
import pandas as pd
import numpy as np

# Prepare columns/variables from the raw dataset 'df'
tb3 = df.copy()

# Define raw FTEs for waves 1 & 2
tb3['fte_before'] = tb3['empft'] + tb3['nmgrs'] + 0.5 * tb3['emppt']
tb3['fte_after'] = tb3['empft2'] + tb3['nmgrs2'] + 0.5 * tb3['emppt2']

# Wave 2 permanently closed stores (status2 == 3) have FTE = 0
tb3.loc[tb3['status2'] == 3, 'fte_after'] = 0.0

# New Jersey categorization by initial starting wage
tb3['nj_wage_group'] = np.nan
tb3.loc[(tb3['state'] == 1) & (tb3['wage_st'] == 4.25), 'nj_wage_group'] = 'low'
tb3.loc[(tb3['state'] == 1) & (tb3['wage_st'] > 4.25) & (tb3['wage_st'] < 5.00), 'nj_wage_group'] = 'mid'
tb3.loc[(tb3['state'] == 1) & (tb3['wage_st'] >= 5.00), 'nj_wage_group'] = 'high'

# Setup columns mapping
# Columns: PA (i), NJ (ii), Diff NJ-PA (iii), NJ-Low (iv), NJ-Mid (v), NJ-High (vi), Low-High (vii), Mid-High (viii)
def get_stats(subset):
    """Helper to return mean and its standard error"""
    valid = subset.dropna()
    n = len(valid)
    if n == 0:
        return np.nan, np.nan
    mean = valid.mean()
    se = valid.std() / np.sqrt(n)
    return mean, se

results_tb3 = {}

# -----------------------------------------------------
# Row 1: FTE employment before, all available
# -----------------------------------------------------
row1_groups = [
    tb3[tb3['state'] == 0]['fte_before'],
    tb3[tb3['state'] == 1]['fte_before'],
    tb3[tb3['nj_wage_group'] == 'low']['fte_before'],
    tb3[tb3['nj_wage_group'] == 'mid']['fte_before'],
    tb3[tb3['nj_wage_group'] == 'high']['fte_before']
]
stats_row1 = [get_stats(g) for g in row1_groups]

# -----------------------------------------------------
# Row 2: FTE employment after, all available
# -----------------------------------------------------
row2_groups = [
    tb3[tb3['state'] == 0]['fte_after'],
    tb3[tb3['state'] == 1]['fte_after'],
    tb3[tb3['nj_wage_group'] == 'low']['fte_after'],
    tb3[tb3['nj_wage_group'] == 'mid']['fte_after'],
    tb3[tb3['nj_wage_group'] == 'high']['fte_after']
]
stats_row2 = [get_stats(g) for g in row2_groups]

# -----------------------------------------------------
# Row 3: Change in mean FTE employment, all available
# -----------------------------------------------------
# Note: This computes the simple difference in means between waves 1 and 2
# (not on the paired sample) as designed in Card & Krueger Table 3

# -----------------------------------------------------
# Row 4: Change in mean, balanced sample
# -----------------------------------------------------
balanced = tb3.dropna(subset=['fte_before', 'fte_after']).copy()
balanced['fte_change'] = balanced['fte_after'] - balanced['fte_before']
row4_groups = [
    balanced[balanced['state'] == 0]['fte_change'],
    balanced[balanced['state'] == 1]['fte_change'],
    balanced[balanced['nj_wage_group'] == 'low']['fte_change'],
    balanced[balanced['nj_wage_group'] == 'mid']['fte_change'],
    balanced[balanced['nj_wage_group'] == 'high']['fte_change']
]
stats_row4 = [get_stats(g) for g in row4_groups]

# -----------------------------------------------------
# Row 5: Change setting FTE at temporarily closed stores to 0
# -----------------------------------------------------
# Temporarily closed (status2 == 2) are set to 0
tb5 = tb3.copy()
tb5.loc[tb5['status2'] == 2, 'fte_after'] = 0.0
balanced_t5 = tb5.dropna(subset=['fte_before', 'fte_after']).copy()
balanced_t5['fte_change'] = balanced_t5['fte_after'] - balanced_t5['fte_before']

row5_groups = [
    balanced_t5[balanced_t5['state'] == 0]['fte_change'],
    balanced_t5[balanced_t5['state'] == 1]['fte_change'],
    balanced_t5[balanced_t5['nj_wage_group'] == 'low']['fte_change'],
    balanced_t5[balanced_t5['nj_wage_group'] == 'mid']['fte_change'],
    balanced_t5[balanced_t5['nj_wage_group'] == 'high']['fte_change']
]
stats_row5 = [get_stats(g) for g in row5_groups]

# Format the stats for presentation
def build_row_data(stats_list):
    # Stats_list elements: [PA, NJ, Low, Mid, High]
    pa_m, pa_s = stats_list[0]
    nj_m, nj_s = stats_list[1]
    low_m, low_s = stats_list[2]
    mid_m, mid_s = stats_list[3]
    high_m, high_s = stats_list[4]

    # Differences
    diff_nj_pa = nj_m - pa_m
    diff_nj_pa_se = np.sqrt(pa_s**2 + nj_s**2)

    diff_lh = low_m - high_m
    diff_lh_se = np.sqrt(low_s**2 + high_s**2)

    diff_mh = mid_m - high_m
    diff_mh_se = np.sqrt(mid_s**2 + high_s**2)

    return [
        f"{pa_m:.2f} ({pa_s:.2f})",
        f"{nj_m:.2f} ({nj_s:.2f})",
        f"{diff_nj_pa:.2f} ({diff_nj_pa_se:.2f})",
        f"{low_m:.2f} ({low_s:.2f})",
        f"{mid_m:.2f} ({mid_s:.2f})",
        f"{high_m:.2f} ({high_s:.2f})",
        f"{diff_lh:.2f} ({diff_lh_se:.2f})",
        f"{diff_mh:.2f} ({diff_mh_se:.2f})"
    ]

# Handle Row 3 separately because it is difference between Wave 2 and Wave 1 means (unpaired)
row3_data = []
for i in range(5):
    m1, s1 = stats_row1[i]
    m2, s2 = stats_row2[i]
    m_diff = m2 - m1
    s_diff = np.sqrt(s1**2 + s2**2)
    row3_data.append((m_diff, s_diff))

# Build Row 3 Columns
pa_m3, pa_s3 = row3_data[0]
nj_m3, nj_s3 = row3_data[1]
low_m3, low_s3 = row3_data[2]
mid_m3, mid_s3 = row3_data[3]
high_m3, high_s3 = row3_data[4]

diff_nj_pa3 = nj_m3 - pa_m3
diff_nj_pa_se3 = np.sqrt(pa_s3**2 + nj_s3**2)

diff_lh3 = low_m3 - high_m3
diff_lh_se3 = np.sqrt(low_s3**2 + high_s3**2)

diff_mh3 = mid_m3 - high_m3
diff_mh_se3 = np.sqrt(mid_s3**2 + high_s3**2)

row3_cols = [
    f"{pa_m3:.2f} ({pa_s3:.2f})",
    f"{nj_m3:.2f} ({nj_s3:.2f})",
    f"{diff_nj_pa3:.2f} ({diff_nj_pa_se3:.2f})",
    f"{low_m3:.2f} ({low_s3:.2f})",
    f"{mid_m3:.2f} ({mid_s3:.2f})",
    f"{high_m3:.2f} ({high_s3:.2f})",
    f"{diff_lh3:.2f} ({diff_lh_se3:.2f})",
    f"{diff_mh3:.2f} ({diff_mh_se3:.2f})"
]

# Create final DataFrame to match paper layout
table_3_df = pd.DataFrame({
    "(i) PA": [build_row_data(stats_row1)[0], build_row_data(stats_row2)[0], row3_cols[0], build_row_data(stats_row4)[0], build_row_data(stats_row5)[0]],
    "(ii) NJ": [build_row_data(stats_row1)[1], build_row_data(stats_row2)[1], row3_cols[1], build_row_data(stats_row4)[1], build_row_data(stats_row5)[1]],
    "(iii) NJ - PA": [build_row_data(stats_row1)[2], build_row_data(stats_row2)[2], row3_cols[2], build_row_data(stats_row4)[2], build_row_data(stats_row5)[2]],
    "(iv) Wage = $4.25": [build_row_data(stats_row1)[3], build_row_data(stats_row2)[3], row3_cols[3], build_row_data(stats_row4)[3], build_row_data(stats_row5)[3]],
    "(v) Wage = $4.26-$4.99": [build_row_data(stats_row1)[4], build_row_data(stats_row2)[4], row3_cols[4], build_row_data(stats_row4)[4], build_row_data(stats_row5)[4]],
    "(vi) Wage >= $5.00": [build_row_data(stats_row1)[5], build_row_data(stats_row2)[5], row3_cols[5], build_row_data(stats_row4)[5], build_row_data(stats_row5)[5]],
    "(vii) Low - High": [build_row_data(stats_row1)[6], build_row_data(stats_row2)[6], row3_cols[6], build_row_data(stats_row4)[6], build_row_data(stats_row5)[6]],
    "(viii) Mid - High": [build_row_data(stats_row1)[7], build_row_data(stats_row2)[7], row3_cols[7], build_row_data(stats_row4)[7], build_row_data(stats_row5)[7]]
}, index=[
    "1. FTE employment before",
    "2. FTE employment after",
    "3. Change in mean FTE",
    "4. Change in mean, balanced",
    "5. Change in mean, temp closed as 0"
])

display(table_3_df)

```markdown
## Robustness Checks for DiD Estimates

Following econometric best practices, we evaluate whether our core treatment effect estimates are sensitive to:
1. **Control Variables:** Adding brand/chain fixed effects.
2. **Clustering Standard Errors:** Clustering standard errors to allow for arbitrary correlation within restaurant chains.
3. **Functional Form:** Using a logarithmic transformation ($
\log(\text{FTE}_{after}) - \log(\text{FTE}_{before})$) to estimate percentage changes rather than absolute changes.
4. **Sample Restriction (Outliers):** Excluding extreme outliers (stores with absolute FTE changes in the top 2.5% tail).
```

In [ ]:
import statsmodels.formula.api as smf

# Ensure we use the main paired employment sample
robust_df = samples['employment'].copy()

# 1. Baseline Model
base_mod = smf.ols("change ~ state", data=robust_df).fit(cov_type='HC3')

# 2. Control Variables: Add Chain Brand Controls
control_mod = smf.ols("change ~ state + C(chain)", data=robust_df).fit(cov_type='HC3')

# 3. Alternative Standard Errors: Clustered by Chain Brand
clustered_mod = smf.ols("change ~ state", data=robust_df).fit(cov_type='cluster', cov_kwds={'groups': robust_df['chain']})

# 4. Functional Form: Log Transformation (approximating percentage changes)
# Filter to strictly positive FTE values to avoid log(0)
log_df = robust_df[(robust_df['employment_before'] > 0) & (robust_df['employment_after'] > 0)].copy()
log_df['log_change'] = np.log(log_df['employment_after']) - np.log(log_df['employment_before'])
log_mod = smf.ols("log_change ~ state", data=log_df).fit(cov_type='HC3')

# 5. Outliers: Exclude top and bottom 2.5% of employment changes
lower_limit = robust_df['change'].quantile(0.025)
upper_limit = robust_df['change'].quantile(0.975)
trimmed_df = robust_df[(robust_df['change'] >= lower_limit) & (robust_df['change'] <= upper_limit)]
trimmed_mod = smf.ols("change ~ state", data=trimmed_df).fit(cov_type='HC3')

# Summarize Robustness Results
robust_summary = pd.DataFrame({
    "Specification": [
        "Baseline (Unadjusted)",
        "With Chain Controls",
        "Clustered by Chain",
        "Log FTE Outcome (Percentage)",
        "Trimmed Sample (No Outliers)"
    ],
    "Estimate (NJ Impact)": [
        base_mod.params['state'],
        control_mod.params['state'],
        clustered_mod.params['state'],
        log_mod.params['state'],
        trimmed_mod.params['state']
    ],
    "Std. Error": [
        base_mod.bse['state'],
        control_mod.bse['state'],
        clustered_mod.bse['state'],
        log_mod.bse['state'],
        trimmed_mod.bse['state']
    ],
    "p-value": [
        base_mod.pvalues['state'],
        control_mod.pvalues['state'],
        clustered_mod.pvalues['state'],
        log_mod.pvalues['state'],
        trimmed_mod.pvalues['state']
    ],
    "N": [
        base_mod.nobs,
        control_mod.nobs,
        clustered_mod.nobs,
        log_mod.nobs,
        trimmed_mod.nobs
    ]
})

display(robust_summary.round(4))

***Interpreting above robustness checks results***
